# AI-Powered Accounting Agent for Invoice Processing

## Introduction

This notebook demonstrates an **end-to-end invoice-processing workflow** that uses:

- a **vision model** for OCR-style text extraction from invoice images,
- a **language model** for structured field extraction,
- and **pandas** to convert the parsed results into three accounting-ready tables.

The notebook is designed for **Google Colab** and reads invoice images directly from the GitHub repository:

- **Repository API endpoint:** `https://api.github.com/repos/MaggieChen7/Contracts/contents`
- **Raw file base:** `https://raw.githubusercontent.com/MaggieChen7/Contracts/main`

The overall goal is to transform a set of invoice images into three directly viewable accounting forms:

1. **Invoice Summary**
2. **Invoice Line Items Detail**
3. **General Ledger Entries**

The three accounting tables output by this notebook are generated **directly inside the notebook**, with **one dedicated code cell per table**, so that each result can be viewed immediately after execution.

---

## Overall Input / Output Diagram

```text
Input: Invoice images stored in the GitHub repository
        ↓
Download raw invoice image URL and metadata
        ↓
Vision model performs OCR-style extraction
        ↓
Structured OCR text
        ↓
Language model parses the OCR text into normalized JSON
        ↓
Three accounting tables are generated:
    1. Invoice Summary
    2. Invoice Line Items Detail
    3. General Ledger Entries
        ↓
Each table is displayed directly inside the notebook
```

---

## End-to-End Workflow Diagram

```text
Section 1  Environment setup and global configuration
        ↓
Section 2  GitHub access utilities
        ↓
Section 3  OCR extraction module
        ↓
Section 4  NLP parsing module
        ↓
Section 5  Table-building helper functions
        ↓
Section 6  Pipeline execution
        ↓
Section 7, 8, 9  Display the three accounting tables
        ↓
Section 10  Final processing summary
```


# Three Accounting Forms Generated by This Notebook

Before running the code, it is useful to clarify **what these three forms are**, **why they matter**, and **what each one contributes** to an accounting workflow.

## 1. Invoice Summary

The **Invoice Summary** table is the high-level management and bookkeeping view of each invoice.  
Each row represents **one invoice**, and the table captures the core header-level information needed for downstream accounting and review.

Typical fields include:

- invoice number,
- invoice date,
- vendor name,
- subtotal,
- tax amount,
- total amount,
- currency.

### Why this form matters
This is the most compact overview table for:

- monitoring payables,
- checking invoice completeness,
- supporting month-end closing,
- and serving as a bridge between source documents and accounting records.

---

## 2. Invoice Line Items Detail

The **Invoice Line Items** table is the transaction-detail view.  
Instead of one row per invoice, it expands each invoice into **multiple rows**, one for each extracted line item.

Typical fields include:

- invoice number,
- line number,
- item description,
- quantity,
- unit price,
- line total.

### Why this form matters
This table is useful for:

- checking the detailed composition of each invoice,
- tracing how totals were formed,
- supporting cost analysis,
- enabling item-level audit trails,
- and providing better visibility into purchases or services billed.

---

## 3. General Ledger Entries

The **General Ledger Entries** table converts invoice information into simplified accounting entries.  
This is the accounting-posting view of the data.

Typical fields include:

- entry date,
- account code,
- account name,
- description,
- debit,
- credit.

### Why this form matters
This table is useful for:

- converting invoice data into bookkeeping-ready records,
- supporting journal preparation,
- showing how source invoices map into accounting logic,
- and making the output more useful for accounting students or operational finance users.

---

## Relationship Among the Three Forms

```
                       Invoice Image
                             ↓
                    Parsed Invoice JSON
                             ↓
 ┌───────────────────────────┬───────────────────────────┬
 │                           │                           │
 ↓                           ↓                           ↓
Invoice Summary        Line Items Detail            GL Entries
 ↓                           ↓                           ↓
Header-Level Review   Transaction-Level Review   Accounting Posting Logic
```

Together, these three forms give the notebook both:

- a **document-processing perspective**, and
- an **accounting-application perspective**.

## Section 1 — Environment Setup and Global Configuration

This code cell prepares the notebook environment.

### What this cell does

It performs four important setup tasks:

1. imports all required Python libraries,
2. suppresses non-critical warnings,
3. defines the API configuration,
4. defines the GitHub source endpoints used later in the notebook.

### Why this cell matters

This cell establishes all global variables and shared dependencies used throughout the rest of the notebook.  
Keeping this setup in one place makes the notebook easier to read, debug, and modify.

### Mini flow

```text
Import libraries
    ↓
Set API parameters
    ↓
Set GitHub paths
    ↓
Environment ready for processing
```


In [1]:
# Import 'time' package to record runtime
import time
start_time = time.time()

# Import the requests library so the notebook can make HTTP API calls.
import requests

# Import json so the notebook can parse JSON responses returned by APIs.
import json

# Import os for general operating-system-related utilities.
import os

# Import base64 so image bytes can be converted into base64 strings for the vision model request.
import base64

# Import pandas for dataframe creation, manipulation, and display.
import pandas as pd

# Import re for regular-expression-based text extraction.
import re

# Import Path in case path-like operations are needed later in a platform-independent way.
from pathlib import Path

# Import datetime so timestamps and date-related values can be handled when needed.
from datetime import datetime

# Import typing helpers for clearer function signatures.
from typing import Dict, List

# Import warnings so warning behavior can be controlled.
import warnings

# Hide non-critical warnings to keep notebook output cleaner for teaching/demo purposes.
warnings.filterwarnings('ignore')

# Keep the API key hard-coded exactly as requested by the user.
API_KEY = "sk-cJvvaaDosK2UQKWvPeyK7HyrRK004ul6L9IN7YdZm00cd7xT"

# Define the chat-completions endpoint used for both OCR and NLP parsing.
BASE_URL = "https://api.bianxie.ai/v1/chat/completions"

# Define the multimodal vision model used for OCR-style extraction.
OCR_MODEL = "qwen3-vl-plus"

# Define the language model used for structured invoice parsing.
NLP_MODEL = "qwen-plus"

# Define the GitHub API endpoint that lists the files in the repository directory.
INVOICE_DIR = "https://api.github.com/repos/MaggieChen7/Contracts/contents"

# Define the GitHub raw-content base URL used to download actual image bytes.
RAW_BASE = "https://raw.githubusercontent.com/MaggieChen7/Contracts/main"

# Print a friendly setup confirmation so the user can confirm the environment is ready.
print("✅ Environment setup complete!")

# Print the GitHub API endpoint so the input source is explicit in notebook output.
print(f"📁 Invoice source (GitHub API): {INVOICE_DIR}")

# Print the raw-file base URL so the download source is also visible.
print(f"🌐 Raw file base: {RAW_BASE}")

# Print the model names so the notebook configuration is transparent.
print(f"🧠 OCR model: {OCR_MODEL}")
print(f"🧠 NLP model: {NLP_MODEL}")

✅ Environment setup complete!
📁 Invoice source (GitHub API): https://api.github.com/repos/MaggieChen7/Contracts/contents
🌐 Raw file base: https://raw.githubusercontent.com/MaggieChen7/Contracts/main
🧠 OCR model: qwen3-vl-plus
🧠 NLP model: qwen-plus


## Section 2 — GitHub Data Access Utilities

This code cell defines helper functions for reading invoice files directly from GitHub.

### What this cell does

It creates reusable functions that:

1. build request headers,
2. read the file list from the GitHub repository,
3. keep only image files,
4. generate raw GitHub image URLs,
5. optionally download image bytes from GitHub when needed.

### Why this cell matters

In the original notebook, the logic still looked partly like a **local-folder workflow**.  
However, the notebook has already moved to a **GitHub-based input source**, so the processing logic should also be fully aligned with that design.

This revised cell fixes that mismatch by ensuring the notebook:

- no longer assumes `Path(...).glob(...)` over a local directory,
- reads invoice files from GitHub correctly,
- and prepares image content directly for API-based OCR.

### Mini flow

```text
List repository files
    ↓
Filter image files
    ↓
Build raw GitHub image URL
    ↓
Image is ready for OCR
```


In [2]:
# Define a helper function that returns the standard headers for the external OCR/NLP API.
def get_headers():
    # Return the authorization header and content-type header as a dictionary.
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }


# Define a helper function that requests the GitHub repository directory listing and keeps image files only.
def list_invoice_files_from_github() -> List[Dict]:
    # Send a GET request to the GitHub contents API endpoint.
    response = requests.get(INVOICE_DIR, timeout=60)

    # Raise an error immediately if GitHub does not return a successful response.
    response.raise_for_status()

    # Convert the JSON response into a Python list of file metadata dictionaries.
    all_files = response.json()

    # Define the file extensions that will be treated as invoice images.
    valid_extensions = (".jpg", ".jpeg", ".png")

    # Keep only files whose names end with one of the supported image extensions.
    image_files = [
        file_info
        for file_info in all_files
        if file_info.get("type") == "file" and file_info.get("name", "").lower().endswith(valid_extensions)
    ]

    # Sort the image files by filename so notebook runs are deterministic and easier to compare.
    image_files = sorted(image_files, key=lambda x: x.get("name", "").lower())

    # Return the filtered and sorted list of image-file metadata dictionaries.
    return image_files


# Define a helper function that converts a GitHub file path into a raw-content URL.
def build_raw_github_url(file_path: str) -> str:
    # Combine the raw GitHub base path with the relative file path.
    return f"{RAW_BASE}/{file_path}"


# Define a helper function that downloads raw image bytes from GitHub using the file path.
def fetch_image_bytes_from_github(file_path: str, max_retries: int = 3) -> bytes:
    # Build the raw GitHub file URL by combining the raw base path with the file path.
    raw_url = build_raw_github_url(file_path)

    # Define browser-like headers to make GitHub raw-file downloads more stable.
    github_headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*"
    }

    # Initialize a variable that will keep the last download exception if all retries fail.
    last_error = None

    # Try the GitHub download several times so temporary network instability does not break the notebook.
    for attempt in range(1, max_retries + 1):
        # Start a try block for the current download attempt.
        try:
            # Send a GET request to download the raw image bytes.
            response = requests.get(raw_url, headers=github_headers, timeout=60)

            # Raise an error if the file download fails.
            response.raise_for_status()

            # Return the raw binary content of the image.
            return response.content

        # Catch any exception so the function can retry before failing completely.
        except Exception as e:
            # Store the current exception so it can be raised later if needed.
            last_error = e

    # Raise the last observed download error after all retry attempts are exhausted.
    raise last_error


# Define a helper function that infers the MIME type from the invoice filename.
def get_image_mime_type(file_name: str) -> str:
    # Convert the filename to lowercase so extension matching is case-insensitive.
    lower_name = file_name.lower()

    # Return the PNG MIME type when the filename ends with .png.
    if lower_name.endswith(".png"):
        return "image/png"

    # Return the JPEG MIME type for .jpg and .jpeg files.
    return "image/jpeg"


# Define a helper function that converts GitHub image bytes into a base64 data URL.
def build_base64_data_url_from_github(file_info: Dict) -> str:
    # Read the relative file path from the GitHub metadata dictionary.
    file_path = file_info["path"]

    # Read the display filename from the GitHub metadata dictionary.
    file_name = file_info["name"]

    # Download the raw image bytes from GitHub.
    image_bytes = fetch_image_bytes_from_github(file_path)

    # Convert the raw image bytes into a base64-encoded text string.
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")

    # Infer the correct MIME type from the invoice filename.
    mime_type = get_image_mime_type(file_name)

    # Return a standard base64 data URL that can be passed directly to the vision model.
    return f"data:{mime_type};base64,{image_base64}"


# Run a lightweight repository check so the user can verify that GitHub access is working.
try:
    # Read the available invoice image files from GitHub.
    github_invoice_files = list_invoice_files_from_github()

    # Print a success message showing how many invoice images were found.
    print(f"✅ GitHub access successful! Found {len(github_invoice_files)} invoice image file(s).")

    # Print up to the first five filenames as a quick sanity check.
    print("🔎 Sample files:", [file_info["name"] for file_info in github_invoice_files[:5]])

    # Print the raw URL for the first file so the actual OCR input format is visible.
    if github_invoice_files:
        print("🌐 Sample raw image URL:", build_raw_github_url(github_invoice_files[0]["path"]))

# Catch any exception so the notebook provides a readable setup message instead of a raw crash.
except Exception as e:
    # Print a readable setup failure message.
    print(f"❌ GitHub access check failed: {e}")


✅ GitHub access successful! Found 24 invoice image file(s).
🔎 Sample files: ['batch1-0001.jpg', 'batch1-0002.jpg', 'batch1-0003.jpg', 'batch1-0004.jpg', 'batch1-0005.jpg']
🌐 Sample raw image URL: https://raw.githubusercontent.com/MaggieChen7/Contracts/main/batch1-0001.jpg


## Section 3 — OCR Extraction Module

This code cell defines the OCR-stage function.

### What this cell does

For each invoice image, the function will:

1. build the raw GitHub image URL,
2. send the image URL to the vision model,
3. ask the model to return structured OCR text,
4. capture both success and failure details,
5. store the result in a consistent dictionary format.

### Why this cell matters

This stage is the bridge between the **raw image** and the **structured invoice text** that the NLP stage needs.  
The OCR prompt is intentionally structured so the next step has a cleaner input to parse.

This revised version also improves debugging by preserving:

- HTTP status code,
- response text snippets,
- the raw image URL used in the request,
- and exception messages.

### Mini flow

```text
Raw GitHub image URL
    ↓
Vision model API request
    ↓
Structured OCR text
    ↓
Standardized OCR result dictionary
```


In [3]:
# Define a function that performs OCR-style extraction on one invoice image from GitHub.
def extract_invoice_with_ocr(file_info: Dict) -> Dict:
    # Start a try block so network or API errors can be captured cleanly.
    try:
        # Read the relative file path from the GitHub metadata dictionary.
        file_path = file_info["path"]

        # Read the display filename from the GitHub metadata dictionary.
        file_name = file_info["name"]

        # Build the public raw GitHub image URL.
        raw_image_url = build_raw_github_url(file_path)

        # Build a base64 data URL from the GitHub image so the API does not need to download the remote file itself.
        image_data_url = build_base64_data_url_from_github(file_info)

        # Define the OCR prompt so the model is guided to return a semi-structured result.
        ocr_prompt = '''Extract all text from this invoice and structure it as:

HEADER: Vendor name | Invoice # | Date | Bill-to
ITEMS: Description | Qty | Unit Price | Total (use | separators)
SUMMARY: Subtotal | Tax (VAT/GST/Sales Tax) | Total | Currency
PAYMENT: PO Number

Return clearly structured sections with pipe-separated columns.'''

        # Build the API payload for the vision model.
        payload = {
            "model": OCR_MODEL,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "text", "text": ocr_prompt},
                    {"type": "image_url", "image_url": {"url": image_data_url}}
                ]
            }],
            "max_tokens": 2000,
            "temperature": 0.1
        }

        # Send the OCR request to the API endpoint.
        response = requests.post(BASE_URL, headers=get_headers(), json=payload, timeout=90)

        # Try to parse the response JSON only when possible.
        try:
            response_json = response.json()
        except Exception:
            response_json = None

        # If the API call succeeds, extract the OCR text and return a success dictionary.
        if response.status_code == 200 and response_json:
            # Read the returned OCR text from the response JSON.
            ocr_content = response_json["choices"][0]["message"]["content"]

            # Convert list-style content into a normal string when necessary.
            if isinstance(ocr_content, list):
                # Join all text fragments returned by the API into one string.
                ocr_text = "\n".join(
                    part.get("text", "")
                    for part in ocr_content
                    if isinstance(part, dict)
                ).strip()
            else:
                # Keep plain-string responses unchanged.
                ocr_text = str(ocr_content).strip()

            # Return a standardized success object that keeps the OCR text and file metadata.
            return {
                "success": True,
                "ocr_text": ocr_text,
                "file_name": file_name,
                "file_path": file_path,
                "raw_image_url": raw_image_url,
                "status_code": response.status_code
            }

        # If the API call fails, return a standardized failure object with richer debugging details.
        else:
            # Build a short response preview for debugging.
            response_preview = response.text[:500]

            # Return a standardized failure object with the HTTP status and response snippet.
            return {
                "success": False,
                "error": f"API Error: {response.status_code}",
                "response_preview": response_preview,
                "file_name": file_name,
                "file_path": file_path,
                "raw_image_url": raw_image_url,
                "status_code": response.status_code
            }

    # Catch any unexpected exception and return it in a standardized failure object.
    except Exception as e:
        # Try to keep filename/path information if available.
        return {
            "success": False,
            "error": f"OCR Error: {str(e)}",
            "file_name": file_info.get("name", "unknown_file"),
            "file_path": file_info.get("path", "unknown_path"),
            "raw_image_url": build_raw_github_url(file_info.get("path", "")) if file_info.get("path") else "",
            "status_code": None
        }


# Print a simple confirmation so the user knows the OCR module is loaded.
print("✅ OCR module loaded")


✅ OCR module loaded


## Section 3A — OCR Test and Error Inspection

This optional debug cell runs OCR on **only the first invoice image**.

### What this cell does

It helps verify whether the OCR API is working before the full batch run starts.

### What to look for

- whether the request succeeds,
- whether the model returns OCR text,
- or whether the API returns an error message that explains the failure.

```text
Take the first invoice image
    ↓
Send one OCR request
    ↓
Print success flag, status code, and response details
```


In [4]:
# Read the invoice file list from GitHub.
invoice_files_for_debug = list_invoice_files_from_github()

# Run the OCR function on only the first invoice image when at least one file exists.
if invoice_files_for_debug:
    # Select the first invoice file as a smoke-test example.
    first_test_file = invoice_files_for_debug[0]

    # Print the name of the smoke-test file.
    print("🧪 OCR smoke test file:", first_test_file["name"])

    # Run OCR on the selected file.
    first_test_result = extract_invoice_with_ocr(first_test_file)

    # Print whether the OCR call succeeded.
    print("success:", first_test_result.get("success"))

    # Print the returned HTTP status code.
    print("status_code:", first_test_result.get("status_code"))

    # Print the raw image URL used in the OCR request.
    print("raw_image_url:", first_test_result.get("raw_image_url"))

    # If OCR succeeded, print a short preview of the OCR text.
    if first_test_result.get("success"):
        print("\nOCR preview:")
        print(first_test_result.get("ocr_text", "")[:1000])

    # Otherwise, print the error and the API response preview for debugging.
    else:
        print("\nerror:", first_test_result.get("error"))
        print("\nresponse_preview:")
        print(first_test_result.get("response_preview", ""))
else:
    # Print a readable message if no invoice files were found.
    print("❌ No invoice files were found for OCR smoke testing.")


🧪 OCR smoke test file: batch1-0001.jpg
success: True
status_code: 200
raw_image_url: https://raw.githubusercontent.com/MaggieChen7/Contracts/main/batch1-0001.jpg

OCR preview:
HEADER: Andrews, Kirby and Valdez | 51109338 | 04/13/2013 | Becker Ltd | 8012 Stewart Summit Apt. 455, North Douglas, AZ 95355  
ITEMS: CLEARANCE! Fast Dell Desktop Computer PC DUAL CORE WINDOWS 10 4/8/16GB RAM | 3.00 | 209.00 | 689.70  
ITEMS: HP T520 Thin Client Computer AMD GX-212JC 1.2GHz 4GB RAM TESTED !!READ BELOW!! | 5.00 | 37.75 | 207.63  
ITEMS: gaming pc desktop computer | 1.00 | 400.00 | 440.00  
ITEMS: 12-Core Gaming Computer Desktop PC Tower Affordable GAMING PC 8GB AMD Vega RGB | 3.00 | 464.89 | 1,534.14  
ITEMS: Custom Build Dell Optiplex 9020 MT i5-4570 3.20GHz Desktop Computer PC | 5.00 | 221.99 | 1,220.95  
ITEMS: Dell Optiplex 990 MT Computer PC Quad Core i7 3.4GHz 16GB 2TB HD Windows 10 Pro | 4.00 | 269.95 | 1,187.78  
ITEMS: Dell Core 2 Duo Desktop Computer | Windows XP Pro | 4GB | 500GB | 5.

## Section 4 — NLP Parsing Module

This code cell defines the function that converts OCR text into normalized invoice JSON.

### What this cell does

For each OCR result, the function will:

1. send the OCR text to the language model,
2. ask the model to normalize field names,
3. request a fixed JSON schema,
4. extract the JSON object from the model response,
5. attach the original filename for traceability.

### Why this cell matters

OCR output is still only **semi-structured text**.  
The purpose of this step is to convert that text into a stable machine-readable structure that can be used for dataframe generation.

### Mini flow

```text
Structured OCR text
    ↓
Language model prompt
    ↓
Normalized JSON
    ↓
Invoice record ready for tables
```


In [5]:
# Define a function that converts OCR text into normalized invoice JSON using the language model.
def parse_invoice_with_nlp(ocr_text: str, invoice_filename: str) -> Dict:
    # Build the NLP prompt and explicitly request a fixed JSON schema.
    nlp_prompt = f'''Parse this invoice. Identify similar terms:
- Tax/VAT/GST/Sales Tax -> tax_amount
- Total/Grand Total -> total_amount
- Qty/Quantity -> quantity

Return JSON:
{{
    "invoice_number": "",
    "invoice_date": "YYYY-MM-DD",
    "vendor_name": "",
    "line_items": [{{"description": "", "quantity": 0, "unit_price": 0, "total_amount": 0}}],
    "subtotal": 0,
    "tax_amount": 0,
    "tax_type": "VAT|GST|Sales Tax|Other",
    "total_amount": 0,
    "currency": "USD"
}}

Invoice OCR text:
{ocr_text}'''

    # Build the payload for the text model.
    payload = {
        "model": NLP_MODEL,
        "messages": [
            {"role": "system", "content": "Extract invoice data and return valid JSON only."},
            {"role": "user", "content": nlp_prompt}
        ],
        "max_tokens": 1500,
        "temperature": 0.1
    }

    # Start a try block so parsing errors can be handled gracefully.
    try:
        # Send the NLP request to the external API.
        response = requests.post(BASE_URL, headers=get_headers(), json=payload, timeout=60)

        # If the API call succeeds, attempt to extract the JSON payload from the model output.
        if response.status_code == 200:
            # Read the raw response text produced by the model.
            response_text = response.json()["choices"][0]["message"]["content"]

            # Use a regular expression to locate the JSON object in the text response.
            json_match = re.search(r"\{.*\}", response_text, re.DOTALL)

            # If JSON-like content is found, parse it and return a success object.
            if json_match:
                # Convert the matched JSON text into a Python dictionary.
                invoice_data = json.loads(json_match.group())

                # Attach the original filename for traceability in later tables.
                invoice_data["filename"] = invoice_filename

                # Return a standardized success dictionary containing the parsed data.
                return {
                    "success": True,
                    "data": invoice_data
                }

            # If no JSON object is found, return a structured failure object.
            else:
                return {
                    "success": False,
                    "error": "Could not extract JSON"
                }

        # If the API call itself fails, return a structured API-error object.
        else:
            return {
                "success": False,
                "error": f"API Error: {response.status_code}"
            }

    # Catch any unexpected exception and return it as a structured NLP error.
    except Exception as e:
        return {
            "success": False,
            "error": f"NLP Error: {str(e)}"
        }


# Print a message so the user knows the NLP module is ready.
print("✅ NLP module loaded")

✅ NLP module loaded


## Section 5 — Shared Table-Building Utilities

This code cell defines reusable functions for generating the three accounting tables.

### What this cell does

It creates three helper functions:

1. `build_invoice_summary_df(...)`
2. `build_line_items_df(...)`
3. `build_gl_entries_df(...)`

### Why this cell matters

In the previous version, all three forms were generated together and immediately exported to files.  
In this revised version, the table-building logic is separated from the display logic so that:

- each final table can be generated in its **own code cell**,
- each result can be displayed directly in Colab,
- and the notebook is easier to teach from and modify later.

In [6]:
# Define a function that builds the invoice-summary dataframe from successful parsed invoices.
def build_invoice_summary_df(successful_invoices: List[Dict]) -> pd.DataFrame:
    # Create an empty list that will hold one summary row per invoice.
    invoice_summaries = []

    # Loop through each successfully parsed invoice result.
    for inv in successful_invoices:
        # Read the parsed invoice dictionary from the standardized result object.
        data = inv["data"]

        # Append one summary row containing header-level invoice information.
        invoice_summaries.append({
            "Invoice_Number": data.get("invoice_number", ""),
            "Invoice_Date": data.get("invoice_date", ""),
            "Vendor_Name": data.get("vendor_name", ""),
            "Subtotal": data.get("subtotal", 0),
            "Tax_Amount": data.get("tax_amount", 0),
            "Tax_Type": data.get("tax_type", "None"),
            "Total_Amount": data.get("total_amount", 0),
            "Currency": data.get("currency", "USD"),
            "Source_File": data.get("filename", "")
        })

    # Convert the list of summary dictionaries into a pandas dataframe and return it.
    return pd.DataFrame(invoice_summaries)


# Define a function that builds the line-items dataframe from successful parsed invoices.
def build_line_items_df(successful_invoices: List[Dict]) -> pd.DataFrame:
    # Create an empty list that will hold one row per invoice line item.
    line_items = []

    # Loop through each successful parsed invoice.
    for inv in successful_invoices:
        # Read the parsed invoice data dictionary.
        data = inv["data"]

        # Loop through the line items inside the invoice and number them starting from 1.
        for idx, item in enumerate(data.get("line_items", []), start=1):
            # Append one line-item row into the accumulator list.
            line_items.append({
                "Invoice_Number": data.get("invoice_number", ""),
                "Line_Number": idx,
                "Description": item.get("description", ""),
                "Quantity": item.get("quantity", 0),
                "Unit_Price": item.get("unit_price", 0),
                "Line_Total": item.get("total_amount", 0),
                "Source_File": data.get("filename", "")
            })

    # Convert the accumulated line-item rows into a dataframe and return it.
    return pd.DataFrame(line_items)

# Define a function that builds simplified but balanced general-ledger entries from successful parsed invoices.
def build_gl_entries_df(successful_invoices: List[Dict]) -> pd.DataFrame:
    # Create an empty list that will hold debit/credit posting rows.
    gl_entries = []

    # Loop through each successful parsed invoice.
    for inv in successful_invoices:
        # Read the parsed invoice data dictionary.
        data = inv["data"]

        # Read the invoice date once so the later posting rows are easier to construct.
        invoice_date = data.get("invoice_date", "")

        # Read the invoice number once for reuse in posting descriptions.
        invoice_number = data.get("invoice_number", "")

        # Read the vendor name once for reuse in posting descriptions.
        vendor_name = data.get("vendor_name", "")

        # Read the subtotal once because it will be used in multiple posting lines.
        subtotal = data.get("subtotal", 0)

        # Read the tax amount once for tax posting logic.
        tax_amount = data.get("tax_amount", 0)

        # Read the total amount from the parsed result if available.
        total_amount = data.get("total_amount", 0)

        # Read the tax type so the tax account name is more informative.
        tax_type = data.get("tax_type", "Tax")

        # If the extracted total amount is missing or invalid, reconstruct it from subtotal plus tax.
        if total_amount is None or total_amount == 0:
            total_amount = subtotal + tax_amount

        # Add the inventory-side debit entry for the pre-tax purchase amount.
        gl_entries.append({
            "Entry_Date": invoice_date,
            "Account_Code": "1400",
            "Account_Name": "Inventory",
            "Description": f"Invoice {invoice_number} from {vendor_name}",
            "Debit": subtotal,
            "Credit": 0
        })

        # If tax exists, record it as recoverable input tax so the journal stays balanced.
        if tax_amount > 0:
            gl_entries.append({
                "Entry_Date": invoice_date,
                "Account_Code": "1300",
                "Account_Name": f"Input {tax_type} Receivable",
                "Description": f"Tax on Invoice {invoice_number}",
                "Debit": tax_amount,
                "Credit": 0
            })

        # Add the accounts-payable credit entry for the full invoice total amount.
        gl_entries.append({
            "Entry_Date": invoice_date,
            "Account_Code": "2100",
            "Account_Name": "Accounts Payable",
            "Description": f"Invoice {invoice_number}",
            "Debit": 0,
            "Credit": total_amount
        })

    # Convert the accumulated GL posting rows into a dataframe and return it.
    return pd.DataFrame(gl_entries)

# Print a message so the user knows the dataframe-building utilities are ready.
print("✅ Table-building utilities loaded")

✅ Table-building utilities loaded


## Section 6 — Main OCR + NLP Processing Pipeline

This code cell runs the complete extraction pipeline across all invoice images found in GitHub.

### What this cell does

It performs the operational processing loop:

1. reads all invoice image files from GitHub,
2. runs OCR on each image,
3. keeps successful OCR results,
4. runs NLP parsing on successful OCR outputs,
5. keeps successful parsed invoices,
6. stores everything in variables that will be reused by the final table cells.

### Why this cell matters

This is the main execution stage of the notebook.  
Once this cell finishes, the notebook has all the structured invoice records needed to generate the three final accounting tables.

In [7]:
# Read the invoice image file list from GitHub so the notebook knows which invoices to process.
invoice_files = list_invoice_files_from_github()

# Print how many invoice images were found before processing starts.
print(f"📊 Found {len(invoice_files)} invoice image(s)\n")

# Create an empty list to store OCR results for every attempted invoice image.
ocr_results = []

# Print a stage header so the notebook output clearly shows that OCR is starting.
print("🔍 STEP 1: OCR Extraction...")

# Loop through the invoice image metadata one by one.
for i, invoice_file in enumerate(invoice_files, start=1):
    # Print progress information so the user can track which file is being processed.
    print(f"  [{i}/{len(invoice_files)}] {invoice_file['name']}...", end="")

    # Run the OCR extraction function on the current GitHub image file.
    ocr_result = extract_invoice_with_ocr(invoice_file)

    # Append the OCR result dictionary to the OCR results list.
    ocr_results.append(ocr_result)

    # Print a success or failure marker for quick visual monitoring.
    print(" ✅" if ocr_result["success"] else " ❌")

    # If OCR failed, print a short one-line debug hint immediately.
    if not ocr_result["success"]:
        print(f"      ↳ {ocr_result.get('error', 'Unknown OCR error')}")
        if ocr_result.get("status_code") is not None:
            print(f"      ↳ HTTP status: {ocr_result.get('status_code')}")
        if ocr_result.get("response_preview"):
            print(f"      ↳ Response preview: {ocr_result['response_preview'][:300]}")

# Keep only OCR results that succeeded so failed OCR records do not move to the NLP stage.
successful_ocr = [result for result in ocr_results if result["success"]]

# Print the OCR success count for quick evaluation.
print(f"✅ OCR: {len(successful_ocr)}/{len(invoice_files)} successful\n")

# Create an empty list to store NLP parsing results.
all_invoices = []

# Print a stage header so the notebook output clearly shows that NLP parsing is starting.
print("🤖 STEP 2: NLP Parsing...")

# Loop through only the successful OCR outputs.
for i, ocr_result in enumerate(successful_ocr, start=1):
    # Read the filename for more readable progress output.
    invoice_filename = ocr_result["file_name"]

    # Print progress information for the NLP step.
    print(f"  [{i}/{len(successful_ocr)}] {invoice_filename}...", end="")

    # Run the NLP parser on the OCR text for the current invoice.
    nlp_result = parse_invoice_with_nlp(ocr_result["ocr_text"], invoice_filename)

    # Append the NLP result dictionary to the invoice-results list.
    all_invoices.append(nlp_result)

    # Print a success or failure marker for quick visual monitoring.
    print(" ✅" if nlp_result["success"] else " ❌")

    # If NLP failed, print a short one-line debug hint immediately.
    if not nlp_result["success"]:
        print(f"      ↳ {nlp_result.get('error', 'Unknown NLP error')}")

# Keep only invoices whose NLP parsing succeeded.
successful_invoices = [inv for inv in all_invoices if inv["success"]]

# Print the NLP success count for quick evaluation.
print(f"✅ NLP: {len(successful_invoices)}/{len(all_invoices)} successful\n")

# Print a short pipeline-ready message so the user knows the final table cells can now be run.
print("📋 Parsed invoice data is ready for table generation.")


📊 Found 24 invoice image(s)

🔍 STEP 1: OCR Extraction...
  [1/24] batch1-0001.jpg... ✅
  [2/24] batch1-0002.jpg... ✅
  [3/24] batch1-0003.jpg... ✅
  [4/24] batch1-0004.jpg... ✅
  [5/24] batch1-0005.jpg... ✅
  [6/24] batch1-0006.jpg... ✅
  [7/24] batch1-0007.jpg... ✅
  [8/24] batch1-0008.jpg... ✅
  [9/24] batch1-0009.jpg... ✅
  [10/24] batch1-0010.jpg... ✅
  [11/24] batch1-0011.jpg... ✅
  [12/24] batch1-0012.jpg... ✅
  [13/24] batch1-0013.jpg... ✅
  [14/24] batch1-0014.jpg... ✅
  [15/24] batch1-0015.jpg... ✅
  [16/24] batch1-0016.jpg... ✅
  [17/24] batch1-0017.jpg... ✅
  [18/24] batch1-0018.jpg... ✅
  [19/24] batch1-0019.jpg... ✅
  [20/24] batch1-0020.jpg... ✅
  [21/24] batch1-0021.jpg... ✅
  [22/24] batch1-0022.jpg... ✅
  [23/24] batch1-0023.jpg... ✅
  [24/24] batch1-0024.jpg... ✅
✅ OCR: 24/24 successful

🤖 STEP 2: NLP Parsing...
  [1/24] batch1-0001.jpg... ✅
  [2/24] batch1-0002.jpg... ✅
  [3/24] batch1-0003.jpg... ✅
  [4/24] batch1-0004.jpg... ✅
  [5/24] batch1-0005.jpg... ✅
  [6/24]

## Section 7 — Generate and Display Table 1: Invoice Summary

This code cell generates the **Invoice Summary** dataframe and displays it directly in the notebook.

### What this cell does

It takes the successful parsed invoice records, converts them into **one row per invoice**, and then displays the result immediately.

### Why this cell matters

This cell replaces the previous “export to results folder” approach for the first form.  
After running this cell, the user can inspect the first final table directly in Colab without opening any external file.

In [8]:
# Check whether there are any successfully parsed invoices before creating the dataframe.
if successful_invoices:
    # Build the invoice-summary dataframe from the parsed invoice records.
    df_summary = build_invoice_summary_df(successful_invoices)
else:
    # Create an empty dataframe so the variable still exists for downstream cells.
    df_summary = pd.DataFrame()

# Print a title line so the notebook output is clearly labeled.
print("📄 TABLE 1 — INVOICE SUMMARY")

# Print the number of rows so the user can quickly verify the output size.
print(f"Rows: {len(df_summary)}")

# Display the dataframe directly inside the notebook.
display(df_summary)

📄 TABLE 1 — INVOICE SUMMARY
Rows: 24


,Invoice_Number,Invoice_Date,Vendor_Name,Subtotal,Tax_Amount,Tax_Type,Total_Amount,Currency,Source_File
0,51109338,2013-04-13,"Andrews, Kirby and Valdez",5640.17,564.02,Sales Tax,6204.19,USD,batch1-0001.jpg
1,12847181,2012-03-03,Fitzpatrick and Sons,6236.77,623.68,Sales Tax,6860.45,USD,batch1-0002.jpg
2,19471831,2014-09-04,Palmer Ltd,40677.81,4067.78,Sales Tax,44745.59,USD,batch1-0003.jpg
3,16273983,2017-04-01,Castillo LLC,744.60,74.46,Sales Tax,819.06,USD,batch1-0004.jpg
4,89969473,2016-10-29,"Deleon, Davila and Allen",725.37,72.54,Sales Tax,797.91,USD,batch1-0005.jpg
5,72128555,2013-12-25,Obrien Group,665.76,66.58,Sales Tax,732.34,USD,batch1-0006.jpg
6,11580833,2019-11-24,"Wood, Simpson and Summers",4671.23,467.12,Sales Tax,5138.35,USD,batch1-0007.jpg
7,74589240,2013-12-08,Haas and Sons,11794.44,1179.44,Sales Tax,12973.88,USD,batch1-0008.jpg
8,81978187,2013-01-17,Johnson Ltd,113.45,11.34,Sales Tax,124.79,USD,batch1-0009.jpg
9,99314100,2014-06-25,"Michael, Farrell and Lee",5975.75,597.58,Other,6573.33,USD,batch1-0010.jpg


## Section 8 — Generate and Display Table 2: Invoice Line Items

This code cell generates the **Invoice Line Items** dataframe and displays it directly in the notebook.

### What this cell does

It expands each parsed invoice into its individual line items so the output becomes **one row per line item**, not one row per invoice.

### Why this cell matters

This provides the detailed transaction-level breakdown that is usually hidden when only header-level invoice data is shown.

In [9]:
# Check whether there are any successfully parsed invoices before creating the dataframe.
if successful_invoices:
    # Build the line-items dataframe from the parsed invoice records.
    df_line_items = build_line_items_df(successful_invoices)
else:
    # Create an empty dataframe so the variable still exists for downstream cells.
    df_line_items = pd.DataFrame()

# Print a title line so the notebook output is clearly labeled.
print("📄 TABLE 2 — INVOICE LINE ITEMS")

# Print the number of rows so the user can quickly verify the output size.
print(f"Rows: {len(df_line_items)}")

# Display the dataframe directly inside the notebook.
display(df_line_items)

📄 TABLE 2 — INVOICE LINE ITEMS
Rows: 100


,Invoice_Number,Line_Number,Description,Quantity,Unit_Price,Line_Total,Source_File
0,51109338,1,CLEARANCE! Fast Dell Desktop Computer PC DUAL ...,3.0,209.00,689.70,batch1-0001.jpg
1,51109338,2,HP T520 Thin Client Computer AMD GX-212JC 1.2G...,5.0,37.75,207.63,batch1-0001.jpg
2,51109338,3,gaming pc desktop computer,1.0,400.00,440.00,batch1-0001.jpg
3,51109338,4,12-Core Gaming Computer Desktop PC Tower Affor...,3.0,464.89,1534.14,batch1-0001.jpg
4,51109338,5,Custom Build Dell Optiplex 9020 MT i5-4570 3.2...,5.0,221.99,1220.95,batch1-0001.jpg
...,...,...,...,...,...,...,...
95,37877758,4,48 Inches Marble Coffee Table Top Inlay with Y...,1.0,3400.00,3740.00,batch1-0022.jpg
96,37877758,5,7'x4' Blue Random Marble Dining Hallway Table ...,3.0,8194.09,27040.50,batch1-0022.jpg
97,33102457,1,Blue Shiny Stone Floral Art Dining Table Top M...,4.0,3960.00,17424.00,batch1-0023.jpg
98,33102457,2,"12"" Marble Side Coffee Table Top Multi Stone F...",3.0,259.77,857.24,batch1-0023.jpg


## Section 9 — Generate and Display Table 3: General Ledger Entries

This code cell generates the **General Ledger Entries** dataframe and displays it directly in the notebook.

### What this cell does

It converts the parsed invoice records into simplified accounting posting lines, including:

- inventory debit entries,
- accounts payable credit entries,
- and tax payable debit entries when tax exists.

### Why this cell matters

This cell makes the notebook more useful from an accounting perspective because it shows how extracted invoice data can be mapped into bookkeeping-style records.

In [10]:
# Check whether there are any successfully parsed invoices before creating the dataframe.
if successful_invoices:
    # Build the general-ledger dataframe from the parsed invoice records.
    df_gl = build_gl_entries_df(successful_invoices)
else:
    # Create an empty dataframe so the variable still exists for downstream cells.
    df_gl = pd.DataFrame()

# Print a title line so the notebook output is clearly labeled.
print("📄 TABLE 3 — GENERAL LEDGER ENTRIES")

# Print the number of rows so the user can quickly verify the output size.
print(f"Rows: {len(df_gl)}")

# Display the dataframe directly inside the notebook.
display(df_gl)

# If the dataframe is not empty, calculate total debits and credits for a simple balancing check.
if not df_gl.empty:
    # Sum the debit column.
    total_debit = df_gl["Debit"].sum()

    # Sum the credit column.
    total_credit = df_gl["Credit"].sum()

    # Print the totals so the user can verify that the journal is balanced.
    print(f"Total Debit:  {total_debit:.2f}")
    print(f"Total Credit: {total_credit:.2f}")

    # Print a simple status message showing whether the journal is balanced.
    if abs(total_debit - total_credit) < 1e-6:
        print("✅ The general ledger entries are balanced.")
    else:
        print("❌ The general ledger entries are NOT balanced.")

📄 TABLE 3 — GENERAL LEDGER ENTRIES
Rows: 72


,Entry_Date,Account_Code,Account_Name,Description,Debit,Credit
0,2013-04-13,1400,Inventory,"Invoice 51109338 from Andrews, Kirby and Valdez",5640.17,0.00
1,2013-04-13,1300,Input Sales Tax Receivable,Tax on Invoice 51109338,564.02,0.00
2,2013-04-13,2100,Accounts Payable,Invoice 51109338,0.00,6204.19
3,2012-03-03,1400,Inventory,Invoice 12847181 from Fitzpatrick and Sons,6236.77,0.00
4,2012-03-03,1300,Input Sales Tax Receivable,Tax on Invoice 12847181,623.68,0.00
...,...,...,...,...,...,...
67,2018-09-18,1300,Input Sales Tax Receivable,Tax on Invoice 33102457,1661.93,0.00
68,2018-09-18,2100,Accounts Payable,Invoice 33102457,0.00,18281.24
69,2014-03-01,1400,Inventory,"Invoice 65028137 from Johnson, Barrett and King",89.95,0.00
70,2014-03-01,1300,Input Sales Tax Receivable,Tax on Invoice 65028137,9.00,0.00


Total Debit:  164703.43
Total Credit: 164703.43
✅ The general ledger entries are balanced.


## Section 10 — Final Processing Summary

This code cell prints a compact run summary so the user can quickly understand the processing outcome.

### What this cell does

It reports:

- number of invoice files found,
- OCR success rate,
- NLP success rate,
- overall success rate,
- and simple aggregate financial totals based on the generated summary dataframe.

### Why this cell matters

This gives a concise end-of-run view that is useful for debugging, demos, teaching, and sanity checking.

In [11]:
# Print a visual divider to make the final summary section easy to spot in notebook output.
print("\n" + "=" * 70)

# Print the final summary title.
print("🏁 PROCESSING SUMMARY")

# Print another divider line for readability.
print("=" * 70)

# Print the total number of invoice files discovered from GitHub.
print(f"Total invoices found: {len(invoice_files)}")

# Print the OCR success count and percentage.
ocr_rate = (len(successful_ocr) / len(invoice_files) * 100) if invoice_files else 0
print(f"OCR success rate: {len(successful_ocr)}/{len(invoice_files)} ({ocr_rate:.1f}%)")

# Print NLP-stage metrics only if OCR produced at least one successful record.
if successful_ocr:
    # Calculate the NLP success percentage using the successful OCR set as the denominator.
    nlp_rate = len(successful_invoices) / len(successful_ocr) * 100

    # Calculate the overall success percentage using all discovered invoice files as the denominator.
    overall_rate = (len(successful_invoices) / len(invoice_files) * 100) if invoice_files else 0

    # Print the NLP success rate.
    print(f"NLP success rate: {len(successful_invoices)}/{len(successful_ocr)} ({nlp_rate:.1f}%)")

    # Print the overall end-to-end success rate.
    print(f"Overall success rate: {overall_rate:.1f}%")

# Print financial totals only if the summary dataframe exists and is not empty.
if 'df_summary' in globals() and not df_summary.empty:
    # Print a small section heading for the financial summary.
    print("\nFinancial Totals:")

    # Print the total transaction value across all invoices.
    print(f"  Total transaction value: ${df_summary['Total_Amount'].sum():,.2f}")

    # Print the total tax amount across all invoices.
    print(f"  Total tax amount: ${df_summary['Tax_Amount'].sum():,.2f}")

    # Print the average invoice value.
    print(f"  Average invoice value: ${df_summary['Total_Amount'].mean():,.2f}")

# Print line-item totals only if the line-items dataframe exists.
if 'df_line_items' in globals():
    # Print how many line items were generated.
    print(f"  Line items processed: {len(df_line_items)}")

# Print GL-entry totals only if the GL dataframe exists.
if 'df_gl' in globals():
    # Print how many GL posting rows were generated.
    print(f"  GL entries created: {len(df_gl)}")

# Print a completion message.
print("\n✅ Processing complete!")

end_time = time.time()
print(f"The whole process took {(end_time- start_time)/60:.2f} minutes")
# Print the closing divider line.
print("=" * 70)



🏁 PROCESSING SUMMARY
Total invoices found: 24
OCR success rate: 24/24 (100.0%)
NLP success rate: 24/24 (100.0%)
Overall success rate: 100.0%

Financial Totals:
  Total transaction value: $164,703.43
  Total tax amount: $14,973.05
  Average invoice value: $6,862.64
  Line items processed: 100
  GL entries created: 72

✅ Processing complete!
The whole process took 7.97 minutes
